In [ ]:
# ============================================================
# Stage 3 — Data Cleaning & Splitting for IoT Capstone
# Produces: train.parquet, val.parquet, test.parquet,
#           unknown_holdout.parquet, dev_sample.parquet
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

# ---------- Paths ----------
BASE = '/content/drive/MyDrive/Capstone_IoT'
RAW = os.path.join(BASE, 'Data/Raw')
PROCESSED = os.path.join(BASE, 'Data/Processed')
os.makedirs(PROCESSED, exist_ok=True)

# ---------- Configuration ----------
RANDOM_SEED = 42
LABEL_COL = 'Label'

# Per-class cap for known classes
MAX_PER_CLASS = 30_000

# Split proportions (train/val/test of the known+benign data)
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

# Development sample size (drawn from train after splitting)
DEV_SAMPLE_SIZE = 50_000

# Classes to hold out entirely as "unknown" for open-set evaluation
UNKNOWN_CLASSES = [
    'BACKDOOR_MALWARE',
    'BROWSERHIJACKING',
    'XSS',
    'RECON-PINGSWEEP',
    'UPLOADING_ATTACK',
]

print("Configuration locked.")
print(f"  Random seed: {RANDOM_SEED}")
print(f"  Per-class cap: {MAX_PER_CLASS:,}")
print(f"  Split: {int(TRAIN_FRAC*100)}/{int(VAL_FRAC*100)}/{int(TEST_FRAC*100)}")
print(f"  Unknown holdout classes: {UNKNOWN_CLASSES}")
print(f"  Output directory: {PROCESSED}")

Mounted at /content/drive
Configuration locked.
  Random seed: 42
  Per-class cap: 30,000
  Split: 70/15/15
  Unknown holdout classes: ['BACKDOOR_MALWARE', 'BROWSERHIJACKING', 'XSS', 'RECON-PINGSWEEP', 'UPLOADING_ATTACK']
  Output directory: /content/drive/MyDrive/Capstone_IoT/Data/Processed


In [ ]:
# ============================================================
# Load all 3 merged CSVs and handle missing values
# ============================================================

print("Loading CSV files...")
all_files = sorted([f for f in os.listdir(RAW) if f.endswith('.csv')])
print(f"  Found: {all_files}")

dfs = []
for f in all_files:
    path = os.path.join(RAW, f)
    print(f"  Loading {f}...")
    dfs.append(pd.read_csv(path))

df = pd.concat(dfs, ignore_index=True)
del dfs  # free memory
print(f"\nCombined shape before cleaning: {df.shape}")

# ---------- NaN handling ----------
nan_before = df.isna().sum().sum()
print(f"NaN count before drop: {nan_before}")

df = df.dropna().reset_index(drop=True)

nan_after = df.isna().sum().sum()
print(f"NaN count after drop: {nan_after}")
print(f"Shape after dropping NaN rows: {df.shape}")
print(f"Rows removed: {nan_before // 2} (each affected row had NaN in both Std and Variance)")

print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

Loading CSV files...
  Found: ['Merged01.csv', 'Merged02.csv', 'Merged05.csv', 'Merged09.csv', 'Merged32.csv', 'Merged63.csv']
  Loading Merged01.csv...
  Loading Merged02.csv...
  Loading Merged05.csv...
  Loading Merged09.csv...
  Loading Merged32.csv...
  Loading Merged63.csv...

Combined shape before cleaning: (4016810, 40)
NaN count before drop: 114
NaN count after drop: 0
Shape after dropping NaN rows: (4016753, 40)
Rows removed: 57 (each affected row had NaN in both Std and Variance)

Memory usage: 1441.4 MB


In [ ]:
# ============================================================
# Class distribution inventory on the combined 6-file dataset
# ============================================================

print(f"Total rows: {len(df):,}")
print(f"Total classes: {df[LABEL_COL].nunique()}")
print()

class_counts = df[LABEL_COL].value_counts()
print("Class distribution (all 34 classes):")
print(class_counts.to_string())

print("\n" + "="*60)
print("Unknown holdout class check:")
print("="*60)
for cls in UNKNOWN_CLASSES:
    count = class_counts.get(cls, 0)
    status = "OK" if count > 0 else "MISSING"
    print(f"  {cls:25s}  {count:>8,}  [{status}]")

print(f"\nTotal unknown holdout rows: {sum(class_counts.get(cls, 0) for cls in UNKNOWN_CLASSES):,}")
print(f"Benign row count: {class_counts.get('BENIGN', 0):,}")

Total rows: 4,016,753
Total classes: 34

Class distribution (all 34 classes):
Label
DDOS-ICMP_FLOOD            614880
DDOS-UDP_FLOOD             460788
DDOS-TCP_FLOOD             383922
DDOS-PSHACK_FLOOD          350307
DDOS-SYN_FLOOD             346680
DDOS-RSTFINFLOOD           345948
DDOS-SYNONYMOUSIP_FLOOD    306944
DOS-UDP_FLOOD              283812
DOS-TCP_FLOOD              229239
DOS-SYN_FLOOD              172830
BENIGN                      94375
MIRAI-GREETH_FLOOD          84601
MIRAI-UDPPLAIN              76112
MIRAI-GREIP_FLOOD           63960
DDOS-ICMP_FRAGMENTATION     38817
VULNERABILITYSCAN           31972
MITM-ARPSPOOFING            26200
DDOS-ACK_FRAGMENTATION      24570
DDOS-UDP_FRAGMENTATION      24514
DNS_SPOOFING                15365
RECON-HOSTDISCOVERY         11565
RECON-OSSCAN                 8230
RECON-PORTSCAN               6982
DOS-HTTP_FLOOD               6203
DDOS-HTTP_FLOOD              2420
DDOS-SLOWLORIS               2015
DICTIONARYBRUTEFORCE         113

In [ ]:
# ============================================================
# Cell 4: Carve unknown holdout, cap known classes, split
# (v2: includes inf/NaN cleaning upstream of the split)
# ============================================================
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(RANDOM_SEED)

# ---------- Step 0: Clean infinity values ----------
# CICIoT2023 contains inf values in rate/statistical columns
# (from zero-duration flows causing divide-by-zero)
inf_count = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
print(f"Infinity values found: {inf_count:,}")

df = df.replace([np.inf, -np.inf], np.nan)
before = len(df)
df = df.dropna().reset_index(drop=True)
after = len(df)
print(f"Rows removed (inf → NaN → drop): {before - after:,}")
print(f"Dataset shape after inf cleaning: {df.shape}")

# ---------- Step 1: Carve out the unknown holdout ----------
unknown_mask = df[LABEL_COL].isin(UNKNOWN_CLASSES)
unknown_df = df[unknown_mask].copy().reset_index(drop=True)
known_df = df[~unknown_mask].copy().reset_index(drop=True)

print(f"\nUnknown holdout: {len(unknown_df):,} rows ({unknown_df[LABEL_COL].nunique()} classes)")
print(f"Known + benign:  {len(known_df):,} rows ({known_df[LABEL_COL].nunique()} classes)")

# ---------- Step 2: Per-class cap on known classes ----------
print(f"\nApplying per-class cap of {MAX_PER_CLASS:,}...")

capped_parts = []
for cls, group in known_df.groupby(LABEL_COL):
    if len(group) > MAX_PER_CLASS:
        sampled = group.sample(n=MAX_PER_CLASS, random_state=RANDOM_SEED)
        capped_parts.append(sampled)
    else:
        capped_parts.append(group)

known_capped = pd.concat(capped_parts, ignore_index=True)
known_capped = known_capped.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)  # reshuffle

print(f"Known + benign after capping: {len(known_capped):,} rows")
print(f"\nClass distribution after cap:")
print(known_capped[LABEL_COL].value_counts().to_string())

# ---------- Step 3: Stratified 70/15/15 split ----------
# First split off test (15%)
train_val, test_df = train_test_split(
    known_capped,
    test_size=TEST_FRAC,
    stratify=known_capped[LABEL_COL],
    random_state=RANDOM_SEED
)

# Then split remaining 85% into train (70/85) and val (15/85)
val_size_relative = VAL_FRAC / (TRAIN_FRAC + VAL_FRAC)
train_df, val_df = train_test_split(
    train_val,
    test_size=val_size_relative,
    stratify=train_val[LABEL_COL],
    random_state=RANDOM_SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"\n{'='*60}")
print("Split summary:")
print(f"{'='*60}")
print(f"  Train: {len(train_df):>8,} rows ({len(train_df)/len(known_capped)*100:.1f}%)")
print(f"  Val:   {len(val_df):>8,} rows ({len(val_df)/len(known_capped)*100:.1f}%)")
print(f"  Test:  {len(test_df):>8,} rows ({len(test_df)/len(known_capped)*100:.1f}%)")
print(f"  Unknown holdout: {len(unknown_df):>8,} rows (separate)")
print(f"  Total: {len(train_df)+len(val_df)+len(test_df)+len(unknown_df):,}")

Infinity values found: 25
Rows removed (inf → NaN → drop): 25
Dataset shape after inf cleaning: (4016728, 40)

Unknown holdout: 1,455 rows (5 classes)
Known + benign:  4,015,273 rows (29 classes)

Applying per-class cap of 30,000...
Known + benign after capping: 610,111 rows

Class distribution after cap:
Label
DOS-SYN_FLOOD              30000
DOS-UDP_FLOOD              30000
DDOS-ICMP_FRAGMENTATION    30000
DDOS-RSTFINFLOOD           30000
MIRAI-GREIP_FLOOD          30000
DDOS-UDP_FLOOD             30000
MIRAI-GREETH_FLOOD         30000
DDOS-SYN_FLOOD             30000
DDOS-ICMP_FLOOD            30000
DDOS-PSHACK_FLOOD          30000
BENIGN                     30000
VULNERABILITYSCAN          30000
DDOS-SYNONYMOUSIP_FLOOD    30000
DOS-TCP_FLOOD              30000
MIRAI-UDPPLAIN             30000
DDOS-TCP_FLOOD             30000
MITM-ARPSPOOFING           26200
DDOS-ACK_FRAGMENTATION     24570
DDOS-UDP_FRAGMENTATION     24514
DNS_SPOOFING               15365
RECON-HOSTDISCOVERY        

In [ ]:
# ============================================================
# Cell 5: Normalize features, carve dev sample, save parquet
# ============================================================
from sklearn.preprocessing import StandardScaler

# Identify feature columns (everything except Label)
FEATURE_COLS = [c for c in train_df.columns if c != LABEL_COL]
print(f"Feature columns: {len(FEATURE_COLS)}")

# ---------- Step 1: Fit scaler on train ONLY ----------
scaler = StandardScaler()
scaler.fit(train_df[FEATURE_COLS])

# ---------- Step 2: Apply to all partitions ----------
def scale_partition(part_df, name):
    scaled = scaler.transform(part_df[FEATURE_COLS])
    out = pd.DataFrame(scaled, columns=FEATURE_COLS)
    out[LABEL_COL] = part_df[LABEL_COL].values
    print(f"  {name:20s} scaled: {out.shape}")
    return out

print("\nApplying z-score normalization (fit on train only)...")
train_scaled = scale_partition(train_df, "train")
val_scaled   = scale_partition(val_df, "val")
test_scaled  = scale_partition(test_df, "test")
unknown_scaled = scale_partition(unknown_df, "unknown_holdout")

# ---------- Step 3: Carve dev sample from train ----------
print(f"\nCarving dev sample ({DEV_SAMPLE_SIZE:,} stratified rows from train)...")
dev_sample, _ = train_test_split(
    train_scaled,
    train_size=DEV_SAMPLE_SIZE,
    stratify=train_scaled[LABEL_COL],
    random_state=RANDOM_SEED
)
dev_sample = dev_sample.reset_index(drop=True)
print(f"  Dev sample:          {dev_sample.shape}")

# ---------- Step 4: Save all partitions as parquet ----------
print(f"\nSaving to {PROCESSED}...")

files_to_save = {
    'train.parquet': train_scaled,
    'val.parquet': val_scaled,
    'test.parquet': test_scaled,
    'unknown_holdout.parquet': unknown_scaled,
    'dev_sample.parquet': dev_sample,
}

for filename, data in files_to_save.items():
    path = os.path.join(PROCESSED, filename)
    data.to_parquet(path, index=False)
    size_mb = os.path.getsize(path) / (1024**2)
    print(f"  {filename:28s} {data.shape}  ({size_mb:.1f} MB)")

# ---------- Step 5: Save the scaler too (for inference later) ----------
import pickle
scaler_path = os.path.join(PROCESSED, 'scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"  scaler.pkl                   (saved for inference)")

print(f"\n{'='*60}")
print("Stage 3 COMPLETE. Ready for modeling.")
print(f"{'='*60}")

Feature columns: 39

Applying z-score normalization (fit on train only)...
  train                scaled: (427077, 40)
  val                  scaled: (91517, 40)
  test                 scaled: (91517, 40)
  unknown_holdout      scaled: (1455, 40)

Carving dev sample (50,000 stratified rows from train)...
  Dev sample:          (50000, 40)

Saving to /content/drive/MyDrive/Capstone_IoT/Data/Processed...
  train.parquet                (427077, 40)  (17.8 MB)
  val.parquet                  (91517, 40)  (4.1 MB)
  test.parquet                 (91517, 40)  (4.1 MB)
  unknown_holdout.parquet      (1455, 40)  (0.1 MB)
  dev_sample.parquet           (50000, 40)  (2.4 MB)
  scaler.pkl                   (saved for inference)

Stage 3 COMPLETE. Ready for modeling.
